# Cart-Pole Swing-up: Energy Shaping + LQR

This notebook follows the same pattern as Russ's examples:

1. build the cart-pole from the Drake/Underactuated URDF,
2. use energy shaping to drive the pendulum to the upright energy level,
3. switch to an LQR balancing controller near the top.

Keep the text short; the point is to feel the controller.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pydrake.all import (
    AddMultibodyPlantSceneGraph,
    DiagramBuilder,
    LeafSystem,
    Linearize,
    LinearQuadraticRegulator,
    MeshcatVisualizer,
    Parser,
    Saturation,
    Simulator,
    StartMeshcat,
    VectorLogSink,
    wrap_to,
)

from underactuated import ConfigureParser, running_as_notebook

np.set_printoptions(formatter={"float": lambda x: "{0:0.4f}".format(x)})

In [ ]:
# Start the visualizer. Run this cell only once; each instance consumes a port.
meshcat = StartMeshcat()

## The controller

For the normalized cart-pole model, collocated partial feedback linearization gives

\[
\ddot{x}=u,\qquad \ddot{\theta}=-u\cos\theta-g\sin\theta.
\]

Use the pendulum energy

\[
E(\theta,\dot\theta)=\frac{1}{2}\dot\theta^2-g\cos\theta,\qquad E^d=g,
\]

and define \(\tilde E = E-E^d\). Then

\[
\dot{\tilde E}=-u\dot\theta\cos\theta.
\]

So the swing-up command is

\[
\ddot{x}^d
=
k_E\dot\theta\cos\theta \tilde E
-k_p x-k_d\dot{x}.
\]

This drives the pole toward the homoclinic orbit. Close to the upright, we switch to LQR.

In [ ]:
UPRIGHT_STATE = np.array([0.0, np.pi, 0.0, 0.0])


def angle_error_about_upright(theta):
    # Return theta - pi, wrapped to [-pi, pi]-like coordinates.
    return wrap_to(theta, 0.0, 2.0 * np.pi) - np.pi


def state_error_about_upright(x):
    xbar = np.asarray(x).copy() - UPRIGHT_STATE
    xbar[1] = angle_error_about_upright(x[1])
    return xbar


def pendulum_energy(x, g=9.81):
    # Unit-length, unit-mass pendulum energy used by the energy shaper.
    theta = x[1]
    theta_dot = x[3]
    return 0.5 * theta_dot**2 - g * np.cos(theta)


def desired_pendulum_energy(g=9.81):
    # Energy of the upright configuration.
    return g

In [ ]:
def BalancingLQR(plant, Q=None, R=None):
    # Compute the LQR gain and Riccati matrix around the upright fixed point.
    context = plant.CreateDefaultContext()
    plant.get_actuation_input_port().FixValue(context, [0.0])
    plant.SetPositionsAndVelocities(context, UPRIGHT_STATE)

    if Q is None:
        Q = np.diag((10.0, 10.0, 1.0, 1.0))
    if R is None:
        R = np.array([[1.0]])

    linearized_plant = Linearize(
        plant,
        context,
        input_port_index=plant.get_actuation_input_port().get_index(),
        output_port_index=plant.get_state_output_port().get_index(),
    )

    K, S = LinearQuadraticRegulator(
        linearized_plant.A(), linearized_plant.B(), Q, R
    )
    return K, S


class CartPoleSwingUpAndBalanceController(LeafSystem):
    def __init__(
        self,
        plant,
        kE=4.0,
        kp=1.0,
        kd=2.0,
        rho=15.0,
        g=9.81,
    ):
        LeafSystem.__init__(self)
        self.DeclareVectorInputPort("state", 4)
        self.DeclareVectorOutputPort("force", 1, self.DoCalcOutput)

        self.K, self.S = BalancingLQR(plant)
        self.kE = kE
        self.kp = kp
        self.kd = kd
        self.rho = rho
        self.g = g

    def lqr_force(self, x):
        xbar = state_error_about_upright(x)
        return float((-self.K @ xbar).item())

    def energy_shaping_force(self, x):
        cart_x, theta, cart_xdot, theta_dot = x
        s = np.sin(theta)
        c = np.cos(theta)

        E = pendulum_energy(x, self.g)
        E_tilde = E - desired_pendulum_energy(self.g)

        # This is the collocated-PFL command: desired cart acceleration.
        xddot_des = (
            self.kE * theta_dot * c * E_tilde
            - self.kp * cart_x
            - self.kd * cart_xdot
        )

        # Convert desired cart acceleration into the actual cart force.
        # For m_c=m_p=l=1:
        #   xddot = [f + sin(theta)(theta_dot^2 + g cos(theta))] / [1 + sin(theta)^2].
        force = (1.0 + s**2) * xddot_des - s * (theta_dot**2 + self.g * c)
        return float(force)

    def DoCalcOutput(self, context, output):
        x = self.get_input_port(0).Eval(context)
        xbar = state_error_about_upright(x)

        # Same spirit as Russ's pendulum example: use LQR inside a quadratic bowl.
        if xbar.dot(self.S.dot(xbar)) < self.rho:
            u = self.lqr_force(x)
        else:
            u = self.energy_shaping_force(x)

        output.SetFromVector([u])

## Build the Drake diagram

The diagram is

\[
\text{cart-pole state}
\rightarrow
\text{hybrid controller}
\rightarrow
\text{saturation}
\rightarrow
\text{cart force}.
\]

The URDF is the same Underactuated cart-pole model used in the balancing examples; here we use the undamped version because the derivation is for an undamped model.

In [ ]:
def make_cartpole_swingup_diagram(
    kE=4.0,
    kp=1.0,
    kd=2.0,
    rho=15.0,
    force_limit=50.0,
):
    builder = DiagramBuilder()

    plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
    parser = Parser(plant)
    ConfigureParser(parser)
    parser.AddModelsFromUrl("package://underactuated/models/undamped_cartpole.urdf")
    plant.Finalize()
    plant.set_name("cartpole")

    controller = builder.AddSystem(
        CartPoleSwingUpAndBalanceController(
            plant,
            kE=kE,
            kp=kp,
            kd=kd,
            rho=rho,
        )
    )
    controller.set_name("swingup_and_balance_controller")

    saturation = builder.AddSystem(
        Saturation(min_value=[-force_limit], max_value=[force_limit])
    )
    saturation.set_name("force_saturation")

    builder.Connect(plant.get_state_output_port(), controller.get_input_port(0))
    builder.Connect(controller.get_output_port(0), saturation.get_input_port(0))
    builder.Connect(saturation.get_output_port(0), plant.get_actuation_input_port())

    state_logger = builder.AddSystem(VectorLogSink(4))
    state_logger.set_name("state_logger")
    builder.Connect(plant.get_state_output_port(), state_logger.get_input_port(0))

    force_logger = builder.AddSystem(VectorLogSink(1))
    force_logger.set_name("force_logger")
    builder.Connect(saturation.get_output_port(0), force_logger.get_input_port(0))

    meshcat.Delete()
    meshcat.Set2dRenderMode(xmin=-3.0, xmax=3.0, ymin=-1.0, ymax=2.5)
    visualizer = MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)
    visualizer.set_name("visualizer")

    diagram = builder.Build()
    return diagram, plant

In [ ]:
def simulate_cartpole(diagram, plant, x0, sim_time=10.0, realtime=False, record=True):
    simulator = Simulator(diagram)
    simulator.set_target_realtime_rate(1.0 if realtime else 0.0)

    context = simulator.get_mutable_context()
    context.SetTime(0.0)

    plant_context = plant.GetMyMutableContextFromRoot(context)
    plant.SetPositionsAndVelocities(plant_context, x0)

    visualizer = diagram.GetSubsystemByName("visualizer")

    if record:
        visualizer.StartRecording(False)

    simulator.Initialize()
    simulator.AdvanceTo(sim_time if running_as_notebook else min(sim_time, 0.1))

    if record:
        visualizer.PublishRecording()
        visualizer.DeleteRecording()

    state_logger = diagram.GetSubsystemByName("state_logger")
    force_logger = diagram.GetSubsystemByName("force_logger")
    return state_logger.FindLog(context), force_logger.FindLog(context)


def plot_cartpole_log(state_log, force_log=None):
    t = state_log.sample_times()
    x = state_log.data()

    plt.figure(figsize=(8, 4))
    plt.plot(t, x[0, :], label="x")
    plt.plot(t, [angle_error_about_upright(th) for th in x[1, :]], label="theta - pi")
    plt.plot(t, x[2, :], label="xdot")
    plt.plot(t, x[3, :], label="thetadot")
    plt.xlabel("time (s)")
    plt.ylabel("state")
    plt.legend()
    plt.grid(True)
    plt.show()

    E = np.array([pendulum_energy(x[:, i]) for i in range(x.shape[1])])
    E_tilde = E - desired_pendulum_energy()

    plt.figure(figsize=(8, 3))
    plt.plot(t, E_tilde)
    plt.xlabel("time (s)")
    plt.ylabel("E - E_desired")
    plt.grid(True)
    plt.show()

    if force_log is not None:
        tf = force_log.sample_times()
        u = force_log.data()[0, :]
        plt.figure(figsize=(8, 3))
        plt.plot(tf, u)
        plt.xlabel("time (s)")
        plt.ylabel("cart force")
        plt.grid(True)
        plt.show()

## Try it

Do not start exactly at `[0, 0, 0, 0]`. That is the boring constant trajectory Russ warns us about: the energy controller has no velocity to pump from.

Start with a tiny perturbation.

In [ ]:
diagram, plant = make_cartpole_swingup_diagram(
    kE=4.0,
    kp=1.0,
    kd=2.0,
    rho=15.0,
    force_limit=50.0,
)

x0 = np.array([0.0, 0.10, 0.0, 0.0])
state_log, force_log = simulate_cartpole(
    diagram,
    plant,
    x0,
    sim_time=10.0,
    realtime=running_as_notebook,
    record=running_as_notebook,
)

plot_cartpole_log(state_log, force_log)

In [ ]:
# A few other initial conditions to play with.
# Rebuild the diagram between trials so each simulation has fresh logs.

initial_conditions = [
    np.array([0.0, 0.10, 0.0, 0.0]),
    np.array([0.5, 0.20, 0.0, 0.0]),
    np.array([-0.5, -0.20, 0.0, 0.0]),
    np.array([0.0, 0.50, 0.0, 0.0]),
]

for x0 in initial_conditions:
    diagram, plant = make_cartpole_swingup_diagram()
    state_log, force_log = simulate_cartpole(
        diagram,
        plant,
        x0,
        sim_time=10.0,
        realtime=False,
        record=False,
    )
    xf = state_log.data()[:, -1]
    print("x0 =", np.around(x0, 3))
    print("xf =", np.around(xf, 3))
    print("final upright error =", np.around(state_error_about_upright(xf), 3))
    print()

## Things to tune

- `kE`: how aggressively we pump energy into the pendulum.
- `kp`, `kd`: how hard we keep the cart near the origin during swing-up.
- `rho`: the size of the LQR switching bowl, \(x^\top Sx < \rho\).
- `force_limit`: actuator saturation.

The controller is deliberately not magic. It is the classical nonlinear-control move:
simplify the equations, shape an energy, then use a local stabilizer to finish the job.